# MANSER AI — خادم مجاني مؤقت

يشغّل هذا الدفتر واجهة API لمحرك MoneyPrinterTurbo داخل Google Colab، ثم ينشئ رابط HTTPS مؤقتاً عبر ngrok. لا تحتاج إلى كمبيوتر. يلزم حساب Google وحساب ngrok مجاني ورمز ngrok الشخصي. الجلسة مؤقتة وتتوقف عند انتهاء Colab؛ هذا ليس استضافة دائمة.

**مهم:** لا تشارك رابط الخادم أو مفتاح API مع الآخرين.

In [ ]:
import os, subprocess, secrets, pathlib, time
from getpass import getpass

REPO = pathlib.Path('/content/MoneyPrinterTurbo')
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/harry0703/MoneyPrinterTurbo.git',str(REPO)],check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-q','uv','pyngrok'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
subprocess.run(['uv','sync','--frozen','--python','3.11'],check=True)
print('تم تثبيت المحرك.')

## 2. إعداد الحماية والرابط

أنشئ حساباً مجانياً في ngrok من https://dashboard.ngrok.com/signup ثم انسخ Auth Token. لا تضعه في GitHub ولا ترسله في المحادثة.

In [ ]:
from pyngrok import ngrok
from getpass import getpass
ngrok.kill()
token = getpass('الصق ngrok Auth Token: ').strip()
if not token: raise ValueError('يلزم رمز ngrok المجاني')
ngrok.set_auth_token(token)
del token

# مفتاح عشوائي لحماية API وملفات الفيديو
api_key = secrets.token_urlsafe(32)
example = (REPO/'config.example.toml').read_text(encoding='utf-8')
config = example.replace('api_key = ""', 'api_key = "'+api_key+'"', 1)
(REPO/'config.toml').write_text(config,encoding='utf-8')
os.environ['CORS_ALLOWED_ORIGINS'] = '*'
print('مفتاح API الخاص بك (انسخه إلى إعدادات مانسر):')
print(api_key)

## 3. تشغيل API وربط مانسر

شغّل الخلية التالية وانتظر ظهور رابط HTTPS. انسخ الرابط كاملاً إلى إعدادات مانسر، ثم ضع مفتاح API الذي ظهر في الخلية السابقة. اترك هذه الجلسة مفتوحة أثناء التوليد.

In [ ]:
import subprocess, time, urllib.request
from pyngrok import ngrok
log = open('/content/manser-api.log','w')
api_proc = subprocess.Popen(['uv','run','--python','3.11','python','main.py'],cwd=REPO,stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
ready=False
for _ in range(60):
    if api_proc.poll() is not None: break
    try:
        urllib.request.urlopen('http://127.0.0.1:8080/docs',timeout=2); ready=True; break
    except Exception: time.sleep(2)
if not ready:
    log.flush(); print(open('/content/manser-api.log').read()[-5000:]); raise RuntimeError('فشل تشغيل API')
tunnel = ngrok.connect(8080,proto='http',bind_tls=True)
print('عنوان الخادم لإدخاله في تطبيق مانسر:')
print(tunnel.public_url)
print('اترك Colab مفتوحاً؛ عند توقف الجلسة سيتوقف الخادم والرابط.')

## 4. استخدام التطبيق

افتح MANSER AI ثم ⚙ الإعدادات. الصق عنوان HTTPS ومفتاح API واحفظهما. بعد ذلك ابدأ إنشاء الفيديو. للحصول على فيديوهات فعلية قد تحتاج إلى إعداد مزود نموذج لغوي ومصدر مواد/مقاطع وصوت داخل إعدادات MoneyPrinterTurbo؛ المجانية تعتمد على حدود الخطط المجانية المتاحة. ملفات الفيديو في جلسة Colab مؤقتة، فحمّلها قبل إيقاف الجلسة.